# F3 · Núcleo algorítmico, eficiencia y programación orientada a objetos

**Avance Fase 3 — Semana 2.** La Fase 2 dejó un pipeline que funciona, hecho de funciones. Esta fase no cambia los datos ni el resultado: reorganiza ese mismo pipeline como un sistema de piezas que se pueden probar, medir y reutilizar.

| Sección | Qué muestra | Criterio de la rúbrica |
|---|---|---|
| 1 | Punto de partida: el pipeline de la Fase 2 | Codificación funcional |
| 2 | Cuándo un paso es una clase y cuándo una función | Diseño estructurado |
| 3 | Las clases: herencia, polimorfismo y encapsulamiento | Programación orientada a objetos |
| 4 | El pipeline y los patrones de diseño | Documentación de arquitectura |
| 5 | Verificación: el resultado es idéntico al de la Fase 2 | Validación |
| 6 | Para qué sirve separar ajustar de transformar | Preprocesamiento |
| 7 | Eficiencia: dos implementaciones medidas (en `F3_eficiencia.ipynb`) | Eficiencia |
| 8 | Validación en tres escenarios (en `tests/test_f3_clases.py`) | Validación |
| 9 | Conclusiones y proyección a la Fase 4 | — |

El código vive en `src/pipeline/` y este notebook lo importa: no hay lógica duplicada.

In [1]:
# Configuración común: rutas del proyecto y módulos del pipeline.
import io

import pandas as pd

from pipeline import config, ordenes
from pipeline.flujo import OrdersPipeline, build_orders_pipeline
from pipeline.transformers import FunctionStep, OutlierFlagger, SupplierAnonymizer, TopSupplierMarker, Transformer

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.float_format", "{:,.1f}".format)
print("Raíz del proyecto:", config.PROJECT_ROOT.name)

Raíz del proyecto: 002_sumativa-1-avance-proyecto-f1-y-f2-40


## 1. Punto de partida: el pipeline de la Fase 2

En la Fase 2 el pipeline es una cadena de funciones de `src/pipeline/ordenes.py`. Cada una hace una sola tarea, recibe un DataFrame y devuelve uno nuevo. Se ejecuta aquí tal como en los notebooks de F2, para tener el resultado de referencia.

In [2]:
raw = ordenes.load_orders(config.RAW_DATA_DIR)

clean = ordenes.normalize_missing(raw)
typed = ordenes.cast_types(clean)
labeled = ordenes.classify_modality(typed)
aggregated = ordenes.aggregate_orders(labeled)
f2_result = ordenes.flag_extreme_amounts(aggregated)

print(f"Entrada: {len(raw):,} líneas · salida: {len(f2_result):,} órdenes y {f2_result.shape[1]} variables")

Entrada: 25,880 líneas · salida: 10,285 órdenes y 14 variables


Funciona, pero el encadenamiento está escrito a mano en cada notebook, y el límite de monto extremo se recalcula cada vez con los datos que se le pasen: no hay forma de calcularlo con un período y aplicarlo a otro.

## 2. ¿Función o clase?

Criterio usado: **un paso es una clase solo si aprende algo de los datos y lo usa después.** Si no necesita recordar nada, sigue siendo una función.

| Paso | ¿Aprende algo de los datos? | Queda como |
|---|---|---|
| `normalize_missing`: unifica los ausentes | No | Función |
| `cast_types`: convierte tipos | No | Función |
| `classify_modality`: crea la modalidad | No: la regla es fija | Función |
| `aggregate_orders`: una fila por orden | No | Función |
| Marcar montos extremos | **Sí: el límite** | Clase `OutlierFlagger` |
| Marcar a los mayores proveedores | **Sí: quiénes son** | Clase `TopSupplierMarker` |
| Anonimizar proveedores | **Sí: el código de cada persona** | Clase `SupplierAnonymizer` |

**Enfoque descartado:** convertir cada función en una clase. Habrían sido clases sin estado, que solo envuelven una función y agregan complejidad sin beneficio.

**Sobre la recursividad:** estos datos no tienen una estructura que la pida. La jerarquía de rubros tiene tres niveles fijos y se recorre con un `groupby`; no hay anidamiento de profundidad variable. Se optó por la división funcional y el enfoque iterativo.

## 3. Las clases

### 3.1 La clase base y la herencia

`Transformer` define lo que todos los pasos comparten: el control de estado, la validación de columnas y la copia defensiva. Cada clase hija solo escribe qué aprende (`_learn`) y cómo lo aplica (`_apply`).

In [3]:
# Herencia: las cuatro clases descienden de Transformer y no repiten el control de estado.
for cls in (OutlierFlagger, TopSupplierMarker, SupplierAnonymizer, FunctionStep):
    own = sorted(name for name in vars(cls) if not name.startswith("__") or name == "__init__")
    print(f"{cls.__name__:<20} hereda de {cls.__bases__[0].__name__} · define: {', '.join(own)}")

print()
print("fit y transform se escriben una sola vez, en la base:", "fit" in vars(Transformer), "·", "fit" in vars(OutlierFlagger))

OutlierFlagger       hereda de Transformer · define: __init__, _abc_impl, _apply, _learn, required_columns
TopSupplierMarker    hereda de Transformer · define: __init__, _abc_impl, _apply, _learn, required_columns, summary
SupplierAnonymizer   hereda de Transformer · define: __init__, _abc_impl, _apply, _learn, summary
FunctionStep         hereda de Transformer · define: __init__, _abc_impl, _apply, _learn, name

fit y transform se escriben una sola vez, en la base: True · False


### 3.2 Encapsulamiento

Lo aprendido queda en atributos internos (con guion bajo). Desde fuera se puede leer, pero no alterar, y no se puede transformar antes de ajustar.

In [4]:
flagger = OutlierFlagger()
print("Antes de ajustar:", flagger, "· aprendido:", flagger.params)

# 1) No se puede transformar antes de ajustar.
try:
    flagger.transform(aggregated)
except RuntimeError as error:
    print("Error esperado:", error)

# 2) Al ajustar, el objeto recuerda el límite.
flagger.fit(aggregated)
print("Después de ajustar:", flagger, f"· límite: ${flagger.params['limit']:,.0f}")

# 3) params entrega una copia: modificarla no altera el estado interno.
copy = flagger.params
copy["limit"] = 0
print(f"Tras intentar cambiarlo desde fuera, el límite sigue en ${flagger.params['limit']:,.0f}")

# 4) transform no modifica los datos que recibe.
flagged = flagger.transform(aggregated)
print("Columna nueva en el resultado:", "monto_extremo" in flagged.columns, "· en la entrada:", "monto_extremo" in aggregated.columns)

Antes de ajustar: OutlierFlagger(sin ajustar) · aprendido: {}
Error esperado: OutlierFlagger: hay que llamar a fit() antes de transform().
Después de ajustar: OutlierFlagger(ajustado) · límite: $74,012,533
Tras intentar cambiarlo desde fuera, el límite sigue en $74,012,533
Columna nueva en el resultado: True · en la entrada: False


### 3.3 Polimorfismo

Los pasos son de clases distintas, pero todos responden a las mismas llamadas. Quien los usa no pregunta de qué tipo es cada uno.

In [5]:
steps = [FunctionStep(ordenes.aggregate_orders), OutlierFlagger(), TopSupplierMarker(top=10)]

current = labeled
for step in steps:
    current = step.fit_transform(current)  # la misma llamada para los tres
    print(f"{step.name:<18} ({type(step).__name__:<17}) -> {len(current):,} filas, {current.shape[1]} columnas · aprendió: {step.summary()}")

aggregate_orders   (FunctionStep     ) -> 10,285 filas, 13 columnas · aprendió: {}
OutlierFlagger     (OutlierFlagger   ) -> 10,285 filas, 14 columnas · aprendió: {'limit': 74012533.27794953, 'orders_seen': 10285}
TopSupplierMarker  (TopSupplierMarker) -> 10,285 filas, 15 columnas · aprendió: {'suppliers': 10, 'share': 19.7}


`summary()` también es polimórfico: cada clase decide qué mostrar de lo aprendido. `SupplierAnonymizer` solo entrega cuántas personas codificó, nunca la correspondencia entre RUT y código.

## 4. El pipeline y los patrones de diseño

`OrdersPipeline` (en `src/pipeline/flujo.py`) recibe la lista de pasos, los ejecuta en orden y deja una bitácora. `build_orders_pipeline()` arma el pipeline de la Fase 2.

In [6]:
pipeline = build_orders_pipeline()
print(pipeline)
f3_result = pipeline.fit_transform(raw)
print(pipeline)
pipeline.log

OrdersPipeline(5 pasos, sin ajustar)


OrdersPipeline(5 pasos, ajustado)


,orden,paso,clase,filas_entrada,filas_salida,columnas_salida,aprendido
0,1,normalize_missing,FunctionStep,25880,25880,80,{}
1,2,cast_types,FunctionStep,25880,25880,80,{}
2,3,classify_modality,FunctionStep,25880,25880,81,{}
3,4,aggregate_orders,FunctionStep,25880,10285,13,{}
4,5,OutlierFlagger,OutlierFlagger,10285,10285,14,"{'limit': 74012533.27794953, 'orders_seen': 10285}"


Todo el flujo se ejecuta con una sola llamada, y la bitácora registra qué entró, qué salió y qué aprendió cada paso.

**Patrones aplicados y qué problema resuelven:**

| Patrón | Dónde | Problema que resuelve |
|---|---|---|
| *Template Method* | `Transformer.fit` y `Transformer.transform` | Todos los pasos necesitan lo mismo antes y después de su tarea: validar columnas, registrar que se ajustó, copiar los datos. Sin el patrón, cada clase repetiría ese control y bastaría olvidarlo en una para tener un error difícil de ver. La base fija el esqueleto y cada hija solo completa `_learn` y `_apply`. |
| *Adapter* | `FunctionStep` | El pipeline solo sabe hablar con objetos que tienen `fit` y `transform`, pero cuatro pasos son funciones. Sin el adaptador habría que convertirlas en clases sin estado o hacer que el pipeline distinguiera funciones de clases. `FunctionStep` les da la interfaz común sin tocarlas. |

**Cohesión y acoplamiento.** Cada módulo hace una cosa: `ordenes.py` tiene las funciones, `transformers.py` los pasos que aprenden y `flujo.py` el encadenamiento. Cambiar la regla de montos extremos toca un solo archivo (`ordenes.py`); agregar un paso nuevo no obliga a modificar el pipeline.

In [7]:
# Agregar un paso no requiere tocar OrdersPipeline: basta con que herede de Transformer.
extended = build_orders_pipeline().add(TopSupplierMarker(top=10))
with_top = extended.fit_transform(raw)
print(extended, "· columnas nuevas:", [column for column in with_top.columns if column not in f3_result.columns])

# Y el pipeline rechaza lo que no cumple el contrato.
try:
    OrdersPipeline([ordenes.cast_types])
except TypeError as error:
    print("Error esperado:", error)

OrdersPipeline(6 pasos, ajustado) · columnas nuevas: ['proveedor_top']
Error esperado: Se esperaba un Transformer y se recibió function.


## 5. Verificación: el resultado es idéntico al de la Fase 2

Es la comprobación central de esta fase: la reorganización no debe cambiar nada. Se compara de dos maneras.

In [8]:
# 1) Contra la cadena de funciones de la Fase 2, en memoria: mismos valores, tipos y orden.
pd.testing.assert_frame_equal(f3_result, f2_result)
print("Idéntico a la cadena de funciones de la Fase 2.")

# 2) Contra el archivo que dejó la Fase 2 en data/processed/.
saved = pd.read_csv(
    config.PROCESSED_DATA_DIR / "ordenes.csv",
    parse_dates=["FechaEnvio"],
    dtype={"Codigo": str, "periodo": str, "CodigoProveedor": str, "CodigoUnidadCompra": str},
)
roundtrip = pd.read_csv(io.StringIO(f3_result.to_csv(index=False)), parse_dates=["FechaEnvio"], dtype={"Codigo": str, "periodo": str, "CodigoProveedor": str, "CodigoUnidadCompra": str})
pd.testing.assert_frame_equal(roundtrip, saved)
print(f"Idéntico a data/processed/ordenes.csv: {len(saved):,} órdenes, {saved.shape[1]} variables.")

# 3) Las ocho reglas de validación de la Fase 2 se siguen cumpliendo.
rules = ordenes.validate_orders(f3_result)
assert rules["cumple"].all()
print(f"Reglas de validación cumplidas: {int(rules['cumple'].sum())} de {len(rules)}.")

Idéntico a la cadena de funciones de la Fase 2.
Idéntico a data/processed/ordenes.csv: 10,285 órdenes, 14 variables.
Reglas de validación cumplidas: 8 de 8.


## 6. Para qué sirve separar ajustar de transformar

Con funciones, el límite de monto extremo se calcula siempre con los mismos datos que se marcan. Con la clase se puede **ajustar con septiembre de 2024 y aplicar a septiembre de 2025**: los dos períodos se miden con la misma vara, la de la línea base anterior al cambio legal.

In [9]:
before = raw[raw["periodo"] == "2024-09"]
after = raw[raw["periodo"] == "2025-09"]

baseline = build_orders_pipeline().add(TopSupplierMarker(top=10))
orders_2024 = baseline.fit_transform(before)  # aprende con 2024
orders_2025 = baseline.transform(after)  # aplica lo aprendido a 2025, sin volver a ajustar

learned = baseline.log.set_index("paso")["aprendido"]
print(f"Límite aprendido con 2024: ${learned['OutlierFlagger']['limit']:,.0f} (con ambos períodos: ${pipeline.log.iloc[-1]['aprendido']['limit']:,.0f})")

amount = ordenes.AMOUNT_COLUMN
pd.DataFrame(
    {
        "órdenes": [len(orders_2024), len(orders_2025)],
        "extremas según el límite de 2024": [int(orders_2024["monto_extremo"].sum()), int(orders_2025["monto_extremo"].sum())],
        "% del monto en los 10 mayores proveedores de 2024": [
            orders_2024.loc[orders_2024["proveedor_top"], amount].sum() / orders_2024[amount].sum() * 100,
            orders_2025.loc[orders_2025["proveedor_top"], amount].sum() / orders_2025[amount].sum() * 100,
        ],
    },
    index=["2024-09", "2025-09"],
)

Límite aprendido con 2024: $40,070,599 (con ambos períodos: $74,012,533)


,órdenes,extremas según el límite de 2024,% del monto en los 10 mayores proveedores de 2024
2024-09,4598,73,22.2
2025-09,5687,111,7.2


Dos lecturas que las funciones de la Fase 2 no permitían:

- Medidas con el límite de 2024, las órdenes extremas de 2025 son más que las de 2024.
- Los diez mayores proveedores de 2024 concentran en 2025 una parte mucho menor del gasto: **los mayores proveedores cambian de un año a otro**. La concentración se mantiene, pero no en las mismas manos.

Son dos meses de una región y los montos son nominales: una diferencia entre ellos no puede atribuirse por completo a la Ley 21.634.

## 7. Eficiencia: dos implementaciones medidas

La medición está en un notebook propio, [`F3_eficiencia.ipynb`](F3_eficiencia.ipynb), con sus pruebas en `tests/test_eficiencia.py`. Compara tres pares de implementaciones de operaciones del pipeline con `timeit` (el mínimo de siete repeticiones) y `tracemalloc`, comprobando antes que ambas versiones dan el mismo resultado:

| Comparación | Versión lenta | Versión del pipeline |
|---|---|---|
| Conversión de montos con coma decimal | `apply` fila por fila | `ordenes.parse_amount`, vectorizada |
| Monto total por proveedor y período | Bucle con `iterrows` | `groupby` |

Las cifras, los gráficos y su interpretación están en ese notebook.

## 8. Validación en tres escenarios

Las clases de esta fase se prueban en `tests/test_f3_clases.py`, en los tres escenarios que pide la rúbrica:

| Escenario | Qué se prueba |
|---|---|
| Caso normal | El pipeline de clases da el mismo resultado que la Fase 2 sobre los datos reales; no pierde ni duplica órdenes; cumple las reglas de calidad |
| Caso límite | Una sola orden; ningún dato ausente; una sola modalidad; ajustar con un conjunto y transformar otro |
| Excepción | Transformar antes de ajustar; columna ausente |

Además comprueban el contrato de las clases: que haya una base y al menos dos hijas, que todas ofrezcan los mismos métodos y que ninguna modifique los datos que recibe. Las secciones 3.2 y 4 de este notebook ya muestran dos de esos errores controlados. Las pruebas se ejecutan con `python -m pytest` y en cada subida al repositorio.

## 9. Conclusiones y proyección a la Fase 4

- El pipeline de la Fase 2 quedó reorganizado sin cambiar su resultado: la comparación con `assert_frame_equal` lo demuestra.
- Solo tres pasos son clases, porque solo ellos aprenden de los datos. El resto sigue como funciones y entra al pipeline por un adaptador.
- Los tres principios están en el código: herencia (cuatro clases sobre `Transformer`), polimorfismo (el pipeline llama a `fit`, `transform` y `summary` sin distinguir clases) y encapsulamiento (estado interno, copia de lo aprendido y control de uso).
- Separar ajustar de transformar ya entregó un resultado nuevo: los mayores proveedores de 2024 no son los de 2025.

**Para la Fase 4:** el análisis de concentración usará este pipeline, informando los indicadores con todas las órdenes y sin las de monto extremo.